# 01 — Synthetic Data Generation

Generates a synthetic per-rep biomechanical feature dataset so the rest of the pipeline (training, evaluation, SHAP, McNemar's test) can be built and validated end-to-end *before/alongside* real video annotation.

This addresses the labelling gap: the Kaggle Workout/Exercises Video dataset labels videos by exercise type, not by posture safety, so a real "unsafe posture" ground truth has to be created (expert annotation or threshold-derived — see `label_rules.py` and `02_extract_real_dataset.ipynb`). This synthetic set is a **fallback / pipeline sanity-check**, not a substitute for real data.

Biomechanical ranges are loosely grounded in the dissertation's literature review (Hanen et al., 2025; Lee et al., 2024; Zhao, 2024):
- Safe squat/deadlift: knee angle bottoms out ~80-100°, hip hinge ~40-70°, trunk lean close to neutral, knee symmetry small (<8°).
- Unsafe: shallower or excessively rounded trunk lean (spinal flexion risk), larger knee valgus/varus asymmetry, faster uncontrolled angular velocity (approximated via larger std/range).

Class distributions deliberately overlap (not cleanly separated) and ~8% of rows get their label flipped, mimicking realistic annotation noise / inter-rater disagreement on borderline reps.

In [ ]:
import sys
sys.path.append('.')

import numpy as np
import pandas as pd
from pathlib import Path

from biomechanics import FEATURE_COLUMNS

RNG = np.random.default_rng(42)

In [ ]:
def _rep(label: int) -> dict:
    """Simulate one rep's aggregated mean/std/range feature vector.
    label: 0 = safe, 1 = unsafe."""

    if label == 0:  # safe posture
        knee_mean = RNG.normal(98, 11)
        knee_std = RNG.normal(19, 4)
        knee_range = RNG.normal(68, 10)
        hip_hinge_mean = RNG.normal(54, 9)
        hip_hinge_std = RNG.normal(13, 3)
        hip_hinge_range = RNG.normal(44, 7)
        trunk_lean_mean = RNG.normal(160, 9)   # close to neutral/upright
        trunk_lean_std = RNG.normal(6, 2)
        trunk_lean_range = RNG.normal(17, 5)
        symmetry_mean = np.abs(RNG.normal(4.5, 2.5))
        symmetry_std = np.abs(RNG.normal(2, 1))
        symmetry_range = np.abs(RNG.normal(6, 2.5))
    else:  # unsafe posture
        knee_mean = RNG.normal(105, 12)
        knee_std = RNG.normal(21, 4.5)
        knee_range = RNG.normal(62, 11)
        hip_hinge_mean = RNG.normal(50, 10)
        hip_hinge_std = RNG.normal(15, 3.5)
        hip_hinge_range = RNG.normal(42, 8)
        trunk_lean_mean = RNG.normal(150, 11)   # more forward/rounded lean
        trunk_lean_std = RNG.normal(9, 3)
        trunk_lean_range = RNG.normal(24, 7)
        symmetry_mean = np.abs(RNG.normal(7.5, 3.5))
        symmetry_std = np.abs(RNG.normal(3, 1.5))
        symmetry_range = np.abs(RNG.normal(9.5, 4))

    knee_L_mean = knee_mean + RNG.normal(0, symmetry_mean / 2)
    knee_R_mean = knee_mean - RNG.normal(0, symmetry_mean / 2)

    return {
        "knee_angle_L_mean": knee_L_mean, "knee_angle_L_std": knee_std + RNG.normal(0, 1),
        "knee_angle_L_range": knee_range + RNG.normal(0, 3),
        "knee_angle_R_mean": knee_R_mean, "knee_angle_R_std": knee_std + RNG.normal(0, 1),
        "knee_angle_R_range": knee_range + RNG.normal(0, 3),
        "knee_angle_mean_mean": knee_mean, "knee_angle_mean_std": knee_std,
        "knee_angle_mean_range": knee_range,
        "hip_hinge_angle_mean": hip_hinge_mean, "hip_hinge_angle_std": hip_hinge_std,
        "hip_hinge_angle_range": hip_hinge_range,
        "trunk_lean_angle_mean": trunk_lean_mean, "trunk_lean_angle_std": trunk_lean_std,
        "trunk_lean_angle_range": trunk_lean_range,
        "knee_symmetry_mean": symmetry_mean, "knee_symmetry_std": symmetry_std,
        "knee_symmetry_range": symmetry_range,
        "n_frames": RNG.integers(18, 45),
    }

In [ ]:
def generate_dataset(n_safe: int = 420, n_unsafe: int = 230, exercise_mix: bool = True) -> pd.DataFrame:
    """Generate a synthetic, class-imbalanced (safe > unsafe, mirroring the real-world
    imbalance Leckey et al. 2024 flag as typical) per-rep dataset."""
    rows, labels = [], []

    for _ in range(n_safe):
        rows.append(_rep(0)); labels.append(0)
    for _ in range(n_unsafe):
        rows.append(_rep(1)); labels.append(1)

    exercises = RNG.choice(["squat", "deadlift"], size=len(rows), p=[0.55, 0.45]) if exercise_mix else ["squat"] * len(rows)

    df = pd.DataFrame(rows)
    df = df[FEATURE_COLUMNS]
    df["exercise"] = exercises
    df["label"] = labels

    # simulate ~8% annotation noise (borderline reps where expert raters would plausibly disagree)
    noise_frac = 0.08
    n_noisy = int(len(df) * noise_frac)
    noisy_idx = RNG.choice(len(df), size=n_noisy, replace=False)
    df.loc[noisy_idx, "label"] = 1 - df.loc[noisy_idx, "label"]

    return df.sample(frac=1.0, random_state=42).reset_index(drop=True)

In [ ]:
df = generate_dataset()
df.head()

In [ ]:
print(df["label"].value_counts(normalize=True).rename("class_balance"))
df["label"].value_counts().plot(kind="bar", title="Safe (0) vs Unsafe (1) rep counts");

In [ ]:
out_path = Path("../data/processed/rep_features.csv")
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)
print(f"Wrote {len(df)} synthetic rep rows to {out_path}")